# **Approaches to interpreting and visually presenting the resulting interpretations for the linear regression algorithm.**

[datablog, 5.12.2024](https://t.me/jdata_blog)

Hi, everyone! This practical notebook was made to inspire you to use regression weights in a versatile and careful way. The notebook is part of the open course "Interpretable AI models" and contains practical tasks that are filled in on the platform.

Enjoy!

### **Limitations on applying the model:**

1. The features $x_i$ must influence the target variable linearly. This follows from the fact that linear regression is applied on the assumption that the target variable depends linearly on the features with constant noise $ϵ$ $y = (w, x) + ϵ$ ([the probabilistic formulation of the problem](https://habr.com/ru/articles/514818/#secProbability)).  Besides that, the noise $ϵ$ itself must be [normally distributed](https://en.wikipedia.org/wiki/Normal_distribution). \
\
If the influence of the features on the target variable is not linear, you can try to fix the situation by transforming the data.

  **For continuous features:** taking the logarithm, taking the square root, taking the sine, the `z-score` transformation, `StandardScaler`, `MinMaxScaler` and [other transformations](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.preprocessing).
  
  **For categorical features:** One-Hot encoding. To avoid linear dependence, the column of one of the categories is dropped, but this is not always done — only if ALL the possible categories the model will handle are present in the dataset (in other words, there are no rows where all the categories of the training set are equal to 0).  \

  You can also combine the approaches. For example, split a continuous feature into quantiles [`pd.cut, pd.qcut`](https://pbpython.com/pandas-qcut-cut.html)

 **To check that the assumption of normally distributed noise holds**, you can look at the residuals of the linear regression. It is precisely from the normal distribution of the noise that MSE follows as the loss function.

2. The independent variables (features) must be linearly independent of each other.


### **Interpretability**

Linear regression belongs to the [interpretable](https://habr.com/ru/articles/744866/) algorithms. Feature importance lies inside the weights of the model, but for this fact to be entirely correct it is important to remember that:

- the data must be brought to a single scale.
- the categorical features must be encoded (for example, via OHE).

**In this notebook** we go through 5 methods for presenting the information that is contained in the weights of the model:

1. Looking at the coefficients directly as a bar plot.
2. Presenting the coefficients as a pie chart.
3. Looking at the relative contribution of the coefficients.
4. Analysis of the model residuals.
5. Building confidence regions.

**Analysing feature contributions is also able to:**

- reveal the reasons for a poor fit;
- build hypotheses for improving the model.

**The methods given in the notebook are meant to help:**
- improve your understanding of the interpretability of the model;
- give practical ways to present the importance of contributions, conveying different informational messages to the audience;
- give practical ways to debug the model based on contributions;

Let us begin.

To start with, let us simply train the model.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import mean_absolute_error

Let us fix the randomness.

In [ ]:
RANDOM_STATE = 42

As the dataset under study we take the [California housing dataset](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset), with the target variable y reflecting the median house value in hundreds of thousands of dollars (US).

In [ ]:
path = 'https://github.com/SadSabrina/open-xai-materials/raw/main/data/housing.csv'
data = pd.read_csv(path, index_col=0)

X = data.drop('MedHouseVal', axis=1)
y = data['MedHouseVal']

X.head()

Let us split the data into a training and a test sample, not forgetting about scaling.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=RANDOM_STATE, test_size=0.25)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Let us train a linear regression and compare its prediction with a baseline one — for every house we will simply predict the mean over the training sample.

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

predictions = lr.predict(X_test_scaled)
base = np.array([y_train.mean()]*len(X_test)) # we will simply predict the mean

print('Baseline algorithm quality (MAE): ', mean_absolute_error(y_test, base))
print('Linear regression quality (MAE): ', mean_absolute_error(y_test, predictions))

In [ ]:
from sklearn.metrics import r2_score

print('Baseline algorithm quality (R2): ', r2_score(y_test, base))
print('Linear regression quality (R2): ', r2_score(y_test, predictions))

Not great, but not terrible either! The regression is built, it is better than the baseline, but still it is not built very well. This is in fact related to the nature of the data. Can this be **revealed and shown**? Let us look at that next.

## **Method 1. Visualizing the coefficients as a bar chart.**

After training the linear regression we have the weights — values that represent, for each feature, the strength of its contribution to the prediction.

The features are stored in the attribute lr.coef_

In [ ]:
labels = X_train.columns #feature names
values = # Your code here — write down the weights

To start with, let us look at the values in a table.

In [ ]:
weights_data = pd.DataFrame(values, index=labels, columns=['weitght'])

weights_data

**Task 1.**

Which feature is the most significant one in the model?

(`Your answer`)

The weights can be either positive or negative. The sign reflects the tendency of the linear relationship.

The first way to present them in a more visually pleasant form is a bar chart, a bar plot. The first option — keeping the tendencies of the relationship:

In [ ]:
plt.figure(figsize=(12, 4))

bar = plt.bar(height=lr.coef_, x=labels)

plt.bar_label(bar, padding=-13, color='black')

plt.title('Linear regression feature importance based on the weights');

The second — visualizing the absolute values:

In [ ]:
plt.figure(figsize=(12, 4))

# Fix the code so that it shows the absolute values of the weights

bar = plt.bar(height=values, x=labels)
plt.bar_label(bar, color='black')

plt.title('Linear regression feature importance based on the weights');

# Method 2. Visualizing the coefficients with a pie chart

A pie chart (pie plot) is a no less illustrative way of visually presenting importances. The plus of such a plot is an excellent representation of the shares of the weights. The minus is that it is not convenient to reflect information about whether the relationship is positive or negative.

**Please note:** a pie plot has to be fed the absolute values of the weights.

In [ ]:
plt.figure(figsize=(8, 8))
patches, texts, pcts = plt.pie(np.abs(lr.coef_), autopct='%1.1f%%', wedgeprops={'linewidth': 3.0, 'edgecolor': 'white'},);

plt.setp(pcts, color='white', fontweight='bold')

plt.legend(X_train.columns);

plt.title('Pie chart of feature importance', fontsize=18);

This way is suitable when it is important for you to *convey the relative significance* of a feature among the others visually. That neatly brings us to the third way — computing this importance head-on.

# Method 3. Computing the relative importance of the features

And so — the relative importance. We saw it above on the marks of the pie chart. The contribution of a feature is computed by normalizing each coefficient of the model by the sum of the absolute values of all the weights.

It can also be visualized afterwards, with a bar plot or with exactly the pie chart above (for which, as we have seen, no preliminary computations are needed). But as a table, sorted in advance, the presentation is illustrative as well.

In [ ]:
#Let us make a slightly more convenient dataframe

# Fix the code so that there are no negative values in the percentages.

weights_data2 = pd.DataFrame([labels, values]).T
weights_data2.columns = ['feature', 'feature_weitght']

weights_sum2 = sum(abs(weights_data2['feature_weitght']))

weights_data2['feature_weitght_normalized'] = weights_data2['feature_weitght'].apply(lambda x: round(x/weights_sum2*100, 2))

In [ ]:
weights_data2.sort_values(by='feature_weitght_normalized', ascending=False)

**Task 2.**

Analyse the relative contributions of the features to the model. Select the statements that apply to the relative values.  

# Method 4. Analysis of the distributions of the residuals

Analysis of the residuals of a regression model is originally applied to assess its quality. More about regression analysis [here](http://www.machinelearning.ru/wiki/index.php?title=%D0%90%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7_%D1%80%D0%B5%D0%B3%D1%80%D0%B5%D1%81%D1%81%D0%B8%D0%BE%D0%BD%D0%BD%D1%8B%D1%85_%D0%BE%D1%81%D1%82%D0%B0%D1%82%D0%BA%D0%BE%D0%B2).

But we will focus on interpretation. The first way is to look at the expected and the predicted values as a scatter plot.

In [ ]:
plt.scatter(predictions, y_test)
plt.title('Scatter plot of the predicted values and the real data');

Here, for example, one can see that on large values the model behaves in a strange way.

The second, no less beautiful way to see this is to overlay the distributions of the predicted and the expected values on each other. For example, like this:

In [ ]:
residuals = # Your code here. Compute the absolute values of the model residuals

plt.hist(predictions, label='Predicted values', bins=25);
plt.hist(y_test, label='Expected values', bins=25, alpha=0.8);

plt.title('Distributions of the predicted values and the real data')
plt.legend();

This way the regions of problematic values are visible even better.

**Task 3.**

Analyse the plot. What is wrong with the model? Select the correct statements in the trainer.

# Method 5. Analysis of confidence regions based on the residuals.

From the previous plots it is clear that somewhere the model does not cope very well. Does this happen everywhere? The analysis of confidence regions **based on the residuals** will help to answer that question.

### **The essence of the method**

The idea of this method is to look at and demonstrate in which regions of the features the model is more confident in its predictions, and in which — less. It is easier to understand this on a classification problem.

**Example:**

Suppose there is a model that predicts, for an online user, the presence (1) or the absence (0) of revenue. Suppose that for our task predicting the absence of revenue is the most important thing. Then we can:

- single out the observations of class 0;
- based on the probabilities produced by the model, split the data into intervals:

 - low confidence (probability of "0" < 0.3).
 - medium confidence (0.3 <= probability of "0" < 0.7).
 - high confidence (probability of "0" >= 0.7).
- look at which objects the model's confidence is low for and run an EDA on them.


**A regression problem.**

For a regression problem everything applies as well. We can split the predicted observations by the quantiles of the absolute values of the residuals. For example:

- class 0 — residuals below the 25th quantile
- class 1 - residuals above the 25th but below the 75th quantile
- class 2 - residuals from the 75th quantile on

Further on, our visualizations will concern the distributions of the features along one axis and of the target variable along the other. An example below:

In [ ]:
residuals = pd.Series(abs(predictions - y_test))

q1 = residuals.quantile(0.25)  # the 25th quantile
q2 = # Your code here — single out the 75th quantile

# Creating the groups based on the residuals
residuals_groups = pd.cut(
    residuals,
    bins=[-float('inf'), q1, q2, float('inf')],  # Quantile ranges
    labels=['group 1: <25%', 'group 2: 25-75%', 'group 3: >75%']  # Group labels
)

In [ ]:
residuals['resid_class'] = residuals_groups

In [ ]:
sns.scatterplot(x=X_test['MedInc'], y=y_test, hue=residuals['resid_class']);

plt.legend();
plt.title('Residual distribution class for the target feature and the MedInc variable');

From the plot one can see that the observations that break out of the linear cloud of the relationship between the two variables have the largest residuals in absolute value. Consequently, either there are outliers and noise in the data, or the assumption of a linear relationship for the feature does not hold — by definition this point will make a high-quality approximation of the target variable by a linear regression difficult.

**Task 5.**

Analyse the plot in the notebook. Which contradiction (or contradictions) gets in the way of the linearity of the relationship, and where is it observed?

That is all, everyone! We have gone through 5 methods for analysing and presenting the weights and the predictions of a regression. I wish you productive work and study with data!  

See you in new posts and practices,  \
Your Data-author!